# StreamTimeline Colab Whisper Server

StreamTimeline 전용 Standalone Whisper Server입니다.

GitHub 저장소를 Clone하지 않으며 Whisper 분석만 담당합니다.

---

## 🚀 Quick Start

### 1. Runtime을 GPU로 변경하세요.

```
Runtime
→ Change runtime type
→ Hardware accelerator
→ GPU
```

### 2. 무료 Ngrok Auth Token을 발급받으세요.

https://dashboard.ngrok.com/get-started/your-authtoken

### 3. 모두 실행을 클릭하세요.

Ngrok Token을 입력하면 Whisper Server가 시작됩니다.

### 4. 마지막 셀에서 출력되는 `Colab API URL`을 StreamTimeline GUI에 붙여넣으세요.

---

## ❓ FAQ

### Q. 매번 Ngrok Token을 입력해야 하나요?

**A. 아닙니다.**

Google Drive에 이 Notebook의 **사본**을 저장하면
Ngrok Token 및 일부 로컬 변수 값을 유지하여
매번 다시 입력하지 않고 사용할 수 있습니다.

**StreamTimeline에서는 사본 사용을 권장합니다.**

단, 사본은 자동으로 업데이트되지 않습니다.

새로운 기능이 추가되거나 문제가 발생한 경우에는
StreamTimeline GUI의 **Google Colab** 버튼을 눌러
최신 Notebook으로 다시 실행한 뒤 새로운 사본을 생성하는 것을 권장합니다.

---

### Q. Colab API URL이 변경되었습니다.

Colab Runtime이 종료되면 새로운 URL이 생성됩니다.
새로 생성된 URL을 StreamTimeline GUI에 다시 입력하면 됩니다.

---

### Q. GitHub 저장소를 Clone해야 하나요?

아닙니다.

이 Notebook은 Standalone 방식으로 동작하며
GitHub Repository를 Clone하지 않습니다.


In [ ]:
import os
import getpass

print("=" * 60)
print("StreamTimeline - Ngrok 설정")
print("=" * 60)
print()
print("Ngrok Auth Token이 필요합니다.")
print("아직 없다면 아래 링크에서 무료로 발급받으세요.")
print()
print("https://dashboard.ngrok.com/get-started/your-authtoken")
print()
print("※ Google Drive에 Notebook 사본을 저장하면")
print("   Ngrok Token을 매번 다시 입력하지 않아도 됩니다.")
print()
NGROK_TOKEN = getpass.getpass("Ngrok Token 입력: ").strip()

if not NGROK_TOKEN:
    raise RuntimeError("Ngrok Token이 입력되지 않았습니다.")

os.environ["NGROK_TOKEN"] = NGROK_TOKEN

print()
print("✓ Ngrok Token이 설정되었습니다.")


In [ ]:
%pip install -q \
fastapi \
uvicorn \
pyngrok \
faster-whisper \
python-multipart


In [ ]:
%%writefile streamtimeline_colab_server.py
import os
import json
import shutil
import threading
from pathlib import Path
from datetime import datetime

from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from faster_whisper import WhisperModel

BASE_DIR = Path("/content/streamtimeline_colab")
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

app = FastAPI()
running_jobs = {}

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


def get_video_dir(title_no):
    path = OUTPUT_DIR / str(title_no)
    path.mkdir(parents=True, exist_ok=True)
    return path


def write_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def update_job(title_no, step, progress, message, done=False, error=None):
    video_dir = get_video_dir(title_no)

    data = {
        "status": "done" if done else "running",
        "step": step,
        "progress": progress,
        "message": message,
        "done": done,
        "error": error,
        "updated_at": datetime.now().isoformat()
    }

    write_json(video_dir / "job.json", data)


def mark_done(title_no, name):
    video_dir = get_video_dir(title_no)
    done_path = video_dir / f"{name}.done"
    done_path.write_text(datetime.now().isoformat(), encoding="utf-8")


def is_done(title_no, name):
    video_dir = get_video_dir(title_no)
    return (video_dir / f"{name}.done").exists()


@app.get("/")
def root():
    return {
        "status": "ready",
        "message": "Standalone StreamTimeline Whisper API is running"
    }


@app.post("/api/v1/upload/start")
def upload_start(
    title_no: str = Form(...),
    title: str = Form(""),
    writer: str = Form(""),
    platform: str = Form("soop")
):
    video_dir = get_video_dir(title_no)
    upload_dir = video_dir / "upload_chunks"

    if upload_dir.exists():
        shutil.rmtree(upload_dir)

    upload_dir.mkdir(parents=True, exist_ok=True)

    metadata = {
        "title_no": title_no,
        "title": title,
        "writer": writer,
        "platform": platform,
        "created_at": datetime.now().isoformat()
    }

    write_json(video_dir / "metadata.json", metadata)
    update_job(title_no, "upload", 65, "Colab 업로드 준비 완료")

    return {
        "status": "ready",
        "title_no": title_no
    }


@app.post("/api/v1/upload/chunk")
async def upload_chunk(
    title_no: str = Form(...),
    index: int = Form(...),
    chunk: UploadFile = File(...)
):
    video_dir = get_video_dir(title_no)
    upload_dir = video_dir / "upload_chunks"
    upload_dir.mkdir(parents=True, exist_ok=True)

    chunk_path = upload_dir / f"chunk_{index:05d}.part"

    data = await chunk.read()

    with open(chunk_path, "wb") as f:
        f.write(data)

    return {
        "status": "uploaded",
        "index": index,
        "size": len(data)
    }


def run_whisper_job(title_no, model_name, language):
    video_dir = get_video_dir(title_no)
    audio_path = video_dir / "audio.wav"
    transcript_path = video_dir / "transcript.json"

    try:
        update_job(title_no, "upload", 70, "업로드 청크를 audio.wav로 합치는 중...")

        if not audio_path.exists() or audio_path.stat().st_size == 0:
            upload_dir = video_dir / "upload_chunks"

            if not upload_dir.exists():
                raise RuntimeError("업로드 청크 폴더가 없습니다.")

            chunk_files = sorted(upload_dir.glob("chunk_*.part"))

            if not chunk_files:
                raise RuntimeError("업로드된 청크가 없습니다.")

            with open(audio_path, "wb") as output:
                for chunk_file in chunk_files:
                    with open(chunk_file, "rb") as f:
                        shutil.copyfileobj(f, output)

            shutil.rmtree(upload_dir)

        mark_done(title_no, "audio")

        if is_done(title_no, "whisper") and transcript_path.exists():
            update_job(title_no, "whisper", 100, "Whisper 완료 파일이 있어 건너뜁니다.", done=True)
            return

        update_job(title_no, "whisper", 75, f"Whisper 모델 로딩 중: {model_name}")

        model = WhisperModel(
            model_name,
            device="cuda",
            compute_type="float16"
        )

        update_job(title_no, "whisper", 80, "Whisper 분석 중...")

        segments, info = model.transcribe(
            str(audio_path),
            language=language,
            vad_filter=True
        )

        transcript_segments = []

        for i, segment in enumerate(segments):
            transcript_segments.append({
                "id": i,
                "start": float(segment.start),
                "end": float(segment.end),
                "text": segment.text.strip()
            })

            if i % 20 == 0:
                update_job(
                    title_no,
                    "whisper",
                    80,
                    f"Whisper 분석 중... segment {i}"
                )

        transcript = {
            "language": language,
            "duration": getattr(info, "duration", None),
            "segments": transcript_segments
        }

        write_json(transcript_path, transcript)
        mark_done(title_no, "whisper")

        update_job(title_no, "whisper", 100, "Whisper 분석이 완료되었습니다.", done=True)

    except Exception as e:
        update_job(
            title_no,
            "whisper",
            0,
            f"Whisper 오류: {e}",
            done=False,
            error=str(e)
        )

    finally:
        running_jobs.pop(str(title_no), None)


@app.post("/api/v1/upload/finish")
def upload_finish(
    title_no: str = Form(...),
    title: str = Form(""),
    writer: str = Form(""),
    platform: str = Form("soop"),
    model: str = Form("large-v3-turbo"),
    language: str = Form("ko")
):
    title_no = str(title_no)

    if title_no in running_jobs:
        return {
            "status": "already_running",
            "job_id": title_no
        }

    update_job(title_no, "queued", 72, "Colab Whisper 작업을 백그라운드로 시작합니다.")

    thread = threading.Thread(
        target=run_whisper_job,
        args=(title_no, model, language),
        daemon=True
    )

    running_jobs[title_no] = thread
    thread.start()

    return {
        "status": "started",
        "job_id": title_no
    }


@app.get("/api/v1/jobs/{job_id}")
def get_job(job_id: str):
    video_dir = get_video_dir(job_id)
    job_path = video_dir / "job.json"

    if not job_path.exists():
        return {
            "status": "not_found",
            "message": "아직 작업 상태가 없습니다.",
            "progress": 0,
            "done": False
        }

    return read_json(job_path)


@app.get("/api/v1/jobs/{job_id}/result")
def get_result(job_id: str):
    video_dir = get_video_dir(job_id)
    transcript_path = video_dir / "transcript.json"

    if not transcript_path.exists():
        raise HTTPException(status_code=404, detail="아직 transcript.json이 없습니다.")

    return {
        "transcript": read_json(transcript_path)
    }


In [ ]:
from pyngrok import ngrok
import os
import threading
import uvicorn
import time

ngrok.set_auth_token(os.environ["NGROK_TOKEN"])

def run_server():
    uvicorn.run(
        "streamtimeline_colab_server:app",
        host="0.0.0.0",
        port=8000,
        reload=False
    )

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

time.sleep(3)
public_url = ngrok.connect(8000, "http")

print("=" * 60)
print("Colab API URL")
print(public_url)
print("=" * 60)
print("이 주소를 StreamTimeline GUI의 Colab API URL에 넣으면 됩니다.")
